# Closed-Loop Causal Refinement — Ground-Truth Simulation (Colab tutorial)

Companion to *"A Closed-Loop Framework for Knowledge-Based Causal Model Refinement Using Process Mining and Human–AI Governance."*

This notebook reproduces the simulation study: a known disease-process simulator generates data with **deliberately injected observation artefacts**, and the governed loop must recover the true causal DAG while **rejecting the artefacts** an ungoverned configuration accepts.

States: **S0** index/on-drug → **S1** CKD → **S2** CVAE → **S3** Death. Trap edges (recorded but not causal): `S2→S0` (drug-feedback decoy), `S2→S1` (reverse coding shift).

## 0. Setup

Installs dependencies and makes the `closed_loop_sim` package importable. **After you push this repo to GitHub, set `REPO_URL`** so Colab can clone it.

In [ ]:
REPO_URL = "https://github.com/<your-org>/closed-loop-simulation-companion.git"  # <-- set after pushing

import os, sys, subprocess
try:
    import closed_loop_sim  # already importable (e.g. running locally)
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "scipy", "pandas", "pydantic", "pm4py", "lifelines", "matplotlib"], check=True)
    folder = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
    if not os.path.exists(folder):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    if os.path.isdir(folder):
        os.chdir(folder); sys.path.insert(0, ".")
    import closed_loop_sim
print("ready:", closed_loop_sim.__doc__)

## 1. The latent truth (DGP)

Each patient walks a continuous-time multi-state model with Weibull proportional-hazards intensities. Exposure (PPI=1, H2B=0) is assigned by a confounded propensity model. The exposure effect is positive on `S0→S1` and **exactly zero on `S1→S2`** (mediation is through CKD *occurrence*, not the CKD→CVAE rate).

In [ ]:
from closed_loop_sim.config import base_cell
from closed_loop_sim.dgp import simulate_population

cfg = base_cell()
pop = simulate_population(2000, cfg, seed=1)
print("PPI prevalence:", round(pop.E.mean(), 3))
print("example trajectory:", pop.histories[0])
print("beta_E  S0->S1 =", cfg.dgp.transitions["S0->S1"].beta_E,
      "|  S1->S2 =", cfg.dgp.transitions["S1->S2"].beta_E)

## 2. The oracle

Because we own the DGP, the simulator is the oracle: true edges, true log-HRs, and the mediated proportion (by g-computation on the simulator).

In [ ]:
from closed_loop_sim.truth import TRUE_EDGES, TRAP_EDGES, mediated_proportion

print("true edges:", sorted(TRUE_EDGES))
print("traps (must be rejected):", sorted(TRAP_EDGES))
med = mediated_proportion(cfg, n=8000, seed=4, horizon=cfg.dgp.t_max)
print("mediated proportion:", round(med["mediated_proportion"], 2))

## 3. The observation process and the artefacts

We record the latent process through visits. PPI users are monitored more (`kappa`), so disease is detected sooner (surveillance bias). Follow-up is truncated at death; the two artefacts are injected after a recorded CVAE.

In [ ]:
from closed_loop_sim.eventlog import build_event_log_data

data = build_event_log_data(6000, cfg, seed=1)
log = data.log
display(log.head(8))
acts = set(log["activity"].unique())
print("artefacts present:", "MedicationChange" in acts, "CKD_recode" in acts)
ckd = log[log.activity == "CKD_recorded"]
print("median CKD recorded time  PPI:", round(ckd[ckd.group=="PPI"].time.median(), 2),
      " H2B:", round(ckd[ckd.group=="H2B"].time.median(), 2), "(PPI earlier = surveillance bias)")

## 4. Discovery: the process map

The recorded log becomes a directly-follows graph (pm4py), mapped to the four state nodes. The two traps appear alongside the true edges.

In [ ]:
from closed_loop_sim.discovery import discover
from closed_loop_sim.visualize import process_map
from IPython.display import Image

disc = discover(log, data.visit_counts)
for (a, b), e in sorted(disc.state_edges.items(), key=lambda x: -x[1].risk):
    tag = "TRAP" if (a, b) in TRAP_EDGES else "true/initial"
    print(f"{a}->{b}  risk={e.risk:.2f}  median={e.median_time:.1f}y  [{tag}]")
process_map(disc)
Image("results/process_map.png")

## 5. Refinement: the three arms

Refinement starts from the initial knowledge graph `{S0→S1, S0→S2, S1→S2}`. The arms differ only in the **gate**: Arm A accepts every frequent proposal; Arm B applies a plausibility gate (backward-edge and administrative-activity rules) that rejects both traps; Arm C is the oracle.

In [ ]:
from closed_loop_sim.refinement import ARMS
from closed_loop_sim.visualize import dag_refinement

for arm, run in ARMS.items():
    res = run(disc)
    print(arm, "traps accepted:", sorted(set(res.edges) & TRAP_EDGES),
          "| Death node:", "S3" in res.nodes)
dag_refinement({arm: set(ARMS[arm](disc).edges) for arm in "ABC"})
Image("results/dag_refinement.png")

## 6. Structural recovery metrics

Arm B has perfect precision and rejects both traps; Arm A's precision is degraded by the accepted artefacts. (Recall 0.83 because the rare `S0→S3` edge is below the 10% threshold — an honest sample-size limitation.)

In [ ]:
from closed_loop_sim.metrics import structural_metrics

for arm in "ABC":
    m = structural_metrics(ARMS[arm](disc))
    print(arm, f"precision={m['precision']:.2f} recall={m['recall']:.2f} "
              f"F1={m['f1']:.2f} SHD={m['shd']} traps={m['n_traps_accepted']}")

## 7. Estimand recovery: the surveillance-bias story

Cause-specific Cox recovers the truth on the **latent** process; surveillance bias inflates the **recorded** estimate; a naive visit-count adjustment **over-corrects** (collider bias). True `beta_E = 0.25`.

In [ ]:
from closed_loop_sim.estimation import estimate

e = estimate(data, cfg.dgp.t_max)
for k in ["s0s1_latent", "s0s1_naive", "s0s1_adjusted"]:
    print(f"{k:14} beta_E = {e[k]['beta_E']:.3f}")

## 8. The full study

Aggregate over replications. (Use a small `M` here; `python -m closed_loop_sim.run_study 500` runs the full archived study.)

In [ ]:
from closed_loop_sim.experiment import run_cell

res = run_cell(cfg, M=30, with_estimation=True)
print("accepts >=1 trap:  Arm A =", res["structural"]["A"]["accept_any_trap_rate"],
      " Arm B =", res["structural"]["B"]["accept_any_trap_rate"])
print("latent beta coverage:", res["estimation"]["s0s1_latent"]["coverage"])
Image("results/figure_simulation.png")   # pre-rendered from the M=500 run

## Summary

| | Arm A (ungated) | Arm B (governed) | Arm C (oracle) |
|---|---|---|---|
| Precision | 0.71 | **1.00** | 1.00 |
| Accepts ≥1 artefact | **100%** | **0%** | 0% |
| Death node added | yes | yes | yes |

The governed loop recovers the true structure (including the Death competing-risk node) and rejects both injected artefacts, while the ungoverned configuration accepts them. The model the framework selects recovers the true cause-specific hazard ratio at near-nominal coverage on the underlying process.

**Next:** see `docs/TUTORIAL.md`, the full spec in `docs/METHODS_SPEC.md`, and the manuscript text in `docs/manuscript_sections.md`. The planned sweeps vary sample size, monitoring differential, timestamp granularity, confounding strength, and artefact rates.